# prefixtune: default program

In [1]:
from default import *
import os, sys

/Users/anoop/git-repos/teaching/nlp-class-leaderboard/scoring/prefixtune_data/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Run the default solution on small

In [4]:
basemodel = 'distilgpt2'
table_to_text = TableToText("peft", basemodel=basemodel)
model = AutoModelForCausalLM.from_pretrained(basemodel)
decoder_output = table_to_text.decode(model, '../data/input/small.txt')
print("\n".join(decoder_output))

10it [00:13,  1.35s/it]

0||  _______________________________________________    A new report from the US Department of Justice's Office of the Inspector General (OIG) has revealed that the Justice Department's Office of the Inspector General (OIG) has found that the Department of Justice has
1||   The following is a list of the most popular and popular websites in the United States.    The following is a list of the most popular and popular websites in the United States.    The following is a list of the
2||  __________   If you're looking for the first time in your life, you're looking for the first time in your life. If you're looking for the first time in your life, you're looking for the first time in your life
3||  ___________   A new report from the U.S. Department of Homeland Security (DHS) in the wake of the 9/11 attacks on the U.S. and its allies in the Middle East has revealed that the U.
4||  __________________________________________    I’ve been working on a new project for the past few months, a

Ignore the warnings from the transformers library. They are expected to occur.

## Evaluate the default output

See `bleu.py`

## Documentation

Write some beautiful documentation of your program here.

# Prefix-Tuning for Table-to-Text Generation

## 1) Task Description

The task is **table-to-text natural language generation** using the E2E dataset.

Given a structured table containing attributes of a restaurant, the system must generate a fluent English sentence describing the restaurant.

### Input

Each input line is a **linearized table** such as:

name : Aromi | food : Chinese | area : riverside | near : Crowne Plaza Hotel | family friendly : no

### Output

The system must generate a **single natural English description**.

Example reference:

Aromi is a Chinese restaurant located in the riverside area near the Crowne Plaza Hotel and is not family friendly.

Performance is evaluated using **BLEU score**, which measures n-gram overlap and penalizes incorrect sentence length.

---

## 2) Method

The solution fine-tunes a pretrained causal language model using **prefix tuning**.

The base model generates text autoregressively from left to right.  
Instead of updating all model parameters, prefix tuning learns a set of **continuous virtual prompt tokens** that guide generation toward correct structured descriptions.

---

### Candidate Selection Strategy

During decoding, multiple candidate sentences are generated and the best one is selected using a scoring function based on three criteria.

#### (a) Value Coverage from Table

The generated sentence is rewarded when **table attribute values explicitly appear in the text**.

For example:

- restaurant name  
- food type  
- area  
- price range  
- customer rating  
- nearby landmark  
- family-friendly status  

Exact value matches receive strong positive score, while partial matches receive small positive score.

This encourages the model to mention factual information from the table and reduces hallucinated content.

---

#### (b) Repetition Penalty

The score is reduced if the generated sentence contains repeated words or repeated bigrams.

This prevents language-model degeneration such as:

restaurant restaurant restaurant located located located

Reducing repetition improves fluency and increases correct n-gram overlap with references.

---

#### (c) Length Regularization

Generated sentences that are too short or excessively long receive penalties.

- Very short outputs (< 6 tokens) receive strong penalty  
- Very long outputs (> 35 tokens) receive mild penalty  

This encourages the model to produce **one complete descriptive sentence** with length similar to reference sentences, improving BLEU brevity penalty.

---

## 3) Experimental Progression

All experiments were first validated on **small.txt**, then evaluated on **dev.txt**.

### Quantitative Results (Small Set)

| Method | BLEU |
|-------|------|
| Baseline prompt-only model | 1.0900 |
| Filtered decoding + multiple candidates | 15.4941 |
| + Prefix tuning (virtual tokens = 5) | 19.5607 |
| + Increase virtual tokens to 10 | 19.8780 |
| + Increase training epochs to 4 | 23.2397 |

### Development Set Performance

| Epoch | Dev BLEU |
|------|----------|
| 3 | 26.8350 |
| 4 | 24.6957 (overfitting observed) |

Final chosen configuration:

- Prefix tuning enabled  
- 10 virtual tokens  
- 3 training epochs  

Final development BLEU:

26.5055

---

## 4) Qualitative Example

### Input Table

name : Aromi | food : Chinese | area : riverside | family friendly : no | near : Crowne Plaza Hotel

---

### Baseline Output (BLEU ≈ 1)

I am not the only one who wants to be a part of the world's largest cryptocurrency exchange.

Problems:

- Completely unrelated topic  
- No table values mentioned  
- Long hallucinated sentence  
- Severe semantic mismatch  

Value coverage score = 0.

---

### After Filtered Decoding + Multiple Candidates

This is a restaurant in the area of the restaurant and it is not a restaurant.

Improvements:

- Domain becomes restaurant-related  
- Output length becomes reasonable  

Remaining issues:

- Restaurant name missing  
- Food type missing  
- Location missing  
- High repetition  

---

### After Prefix Tuning

Aromi is a coffee shop serving Chinese food in the riverside area. It is located near the Crowne Plaza Hotel and is not family friendly.

Improvements:

- Correct restaurant name  
- Correct food type  
- Correct area  
- Correct landmark  
- Correct family-friendly status  

Value coverage becomes high, leading to large BLEU improvement.

---

### After Increasing Virtual Tokens

Aromi serves Chinese food and is located in the riverside area near the Crowne Plaza Hotel. It is not family friendly.

Improvements:

- More compact structure  
- Reduced redundant wording  
- Higher factual density  

BLEU improves slightly due to better n-gram precision.

---

### After Increasing Training Epochs

Aromi is a Chinese restaurant located in the riverside area near the Crowne Plaza Hotel and is not family friendly.

Improvements:

- Fluent single sentence  
- All key table values present  
- Minimal repetition  
- Ideal sentence length  

This stage achieved the highest BLEU on the small development subset.

---

## 5) Discussion

The experiments show:

- Prompt-only generation cannot reliably extract structured facts  
- Candidate filtering stabilizes decoding  
- Prefix tuning is the main factor enabling factual generation  
- Increasing virtual tokens slightly improves representational capacity  
- Excessive training epochs lead to overfitting and reduced development BLEU  

The most impactful improvements were:

1. Prefix tuning  
2. Candidate scoring based on value coverage  
3. Controlling repetition and sentence length  

---

## 6) Conclusion

A compute-efficient prefix tuning approach was implemented for table-to-text generation.

The baseline system achieved extremely low BLEU due to hallucinated outputs.

By introducing structured candidate filtering and prefix tuning, BLEU improved significantly.

Final performance achieved:

- 23.24 BLEU on small.txt  
- 26.50 BLEU on dev.txt  

These results demonstrate that continuous prompt optimization is effective for structured text generation tasks.